# Project Bài 8 · Phân khúc khách tại mốc 10/09/2011

**Học máy cơ bản · Bài 8 · PGS.TS Nguyễn Văn Hậu**

**Tình huống.** Đầu tháng 9/2011, nhà bán lẻ muốn chia khách thành vài nhóm để lên kế hoạch chăm sóc cho mùa cuối năm. Bạn chỉ được dùng dữ liệu **trước 10/09/2011**. Bạn tự tính RFM từ dòng hoá đơn, tự viết thuật toán Lloyd, chọn K có căn cứ, đọc hồ sơ cụm và viết đề xuất có điều kiện.

**Dữ liệu.** `data/online_retail_lines.csv.gz`: 541.909 dòng hoá đơn (Online Retail, UCI, CC BY 4.0). Tại mốc này có 3.370 khách.

| Mốc | Bước | Ô tự kiểm |
|---|---|---|
| M1 | 1: RFM tại 10/09/2011 | số khách; R, F, M của khách 12755 |
| M2 | 2: log1p + chuẩn hoá | trung bình 0, phương sai 1, J với một tâm = n·d |
| M3 | 3: `gan`, `cap_nhat`, `tinh_J`, `lloyd` | sáu điểm: 39 → 12 → 93/8 → 22/3; trùng KMeans cùng khởi tạo; J không tăng |
| M4 | 4 (khá): chọn K bằng inertia, silhouette, ổn định | silhouette K = 2 |
| M5 | 5: hồ sơ K = 4 và báo cáo | cỡ cụm |

**Ba mức.** *Đạt (5 điểm)*: bước 1, 2, 3, 5 và báo cáo một trang. *Khá (+3)*: bước 4. *Nâng cao (+2)*: A (k-means++ tự viết) hoặc B (khách đổi cụm từ tháng 9 sang tháng 12).

**Luật chơi.** Bước 3 không dùng `KMeans` để tính; chỉ dùng để đối chiếu ở ô có ghi rõ. Không dùng dữ liệu từ 10/09/2011 trở đi, trừ phần Nâng cao B. Không có “nhãn đúng”: đừng viết “cụm đúng”, hãy viết “cụm tìm được, với căn cứ …”.

In [ ]:
# Thiết lập: chạy ô này trước
from pathlib import Path
import numpy as np, pandas as pd
from sklearn.preprocessing import StandardScaler
from sklearn.cluster import KMeans
from sklearn.metrics import silhouette_score, adjusted_rand_score

ROOT = Path.cwd()
assert (ROOT / 'data' / 'online_retail_lines.csv.gz').exists(), 'Mở notebook từ thư mục bài (có thư mục data/).'
dong = pd.read_csv(ROOT / 'data' / 'online_retail_lines.csv.gz', dtype={'InvoiceNo': str, 'StockCode': str, 'Description': str, 'Country': str},
                   keep_default_na=False, na_values={'CustomerID': [''], 'Description': ['']}, parse_dates=['InvoiceDate'])
MOC = pd.Timestamp('2011-09-10')

def kiem(ten, dieu_kien, goi_y=''):
    # Ô tự kiểm: in ✅ khi đúng; báo lỗi kèm gợi ý khi sai.
    if not dieu_kien:
        raise AssertionError(f'❌ {ten}. Gợi ý: {goi_y}')
    print(f'✅ {ten}')
print(len(dong), 'dòng')

## Bước 1 · RFM tại mốc 10/09/2011

Giữ dòng mua hợp lệ (có CustomerID, Quantity > 0, UnitPrice > 0, mã hoá đơn không bắt đầu bằng C) **và** `InvoiceDate < MOC`. R = số ngày lịch từ ngày mua cuối tới MOC; F = số InvoiceNo khác nhau; M = tổng Quantity × UnitPrice. Một dòng của bảng `rfm` là một khách (chỉ số là CustomerID kiểu int), ba cột theo thứ tự R, F, M.

In [ ]:
def tinh_rfm(dong, moc):
    # TODO: lọc dòng mua hợp lệ trước mốc, gom theo CustomerID, tính R (ngày lịch), F, M; trả DataFrame cột R, F, M
    raise NotImplementedError('TODO bước 1: tinh_rfm')
rfm = tinh_rfm(dong, MOC)
print(len(rfm), 'khách'); print(rfm.describe().round(1))

In [ ]:
# TỰ KIỂM · bước 1
kiem('số khách tại mốc = 3.370', len(rfm) == 3370, 'nhớ lọc InvoiceDate < MOC và bỏ hoá đơn huỷ')
kiem('khách 12755: R = 190, F = 2, M = 3811.95', tuple(rfm.loc[12755, ['R', 'F']]) == (190, 2) and round(rfm.loc[12755, 'M'], 2) == 3811.95,
     'R tính bằng ngày lịch: chuẩn hoá ngày mua cuối về 0 giờ rồi trừ')

## Bước 2 · log1p rồi chuẩn hoá

Áp `np.log1p` cho ba cột, rồi `StandardScaler` (fit trên bảng của mốc này). Tính tỷ trọng phương sai của ba cột trước và sau. Gọi kết quả là `X` (mảng numpy n × 3).

In [ ]:
# TODO: sc = StandardScaler() fit trên log1p(rfm); X = sc.transform(...)
raise NotImplementedError('TODO bước 2: tạo sc và X')
print('tỷ trọng phương sai, thang gốc:', (rfm.var(ddof=0) / rfm.var(ddof=0).sum()).round(4).tolist())
print('tỷ trọng phương sai, sau biến đổi:', (X.var(0) / X.var(0).sum()).round(4).tolist())

In [ ]:
# TỰ KIỂM · bước 2
kiem('trung bình 0, phương sai 1', np.allclose(X.mean(0), 0, atol=1e-10) and np.allclose(X.var(0), 1), 'StandardScaler dùng độ lệch chuẩn chia n')
kiem('J với một tâm = n·d', np.isclose((X ** 2).sum(), X.shape[0] * 3))

## Bước 3 · Tự viết Lloyd

Viết bốn hàm (chỉ dùng numpy):
- `gan(X, C)`: nhãn của tâm gần nhất cho mỗi điểm (hoà → tâm chỉ số nhỏ, như `argmin`);
- `cap_nhat(X, nhan, K)`: K tâm mới = trung bình theo nhóm;
- `tinh_J(X, nhan, C)`: tổng bình phương khoảng cách tới tâm của nhóm;
- `lloyd(X, C0)`: lặp gán – cập nhật tới khi **không điểm nào đổi nhóm**; trả `(nhan, C, lich_su_J)` với `lich_su_J` là J sau **mỗi nửa bước** (gán, cập nhật, gán, …).

In [ ]:
def gan(X, C):
    # TODO: ma trận d² (n × K) rồi argmin theo trục 1
    raise NotImplementedError('TODO bước 3: gan')

def cap_nhat(X, nhan, K):
    # TODO: trung bình các điểm của từng nhóm
    raise NotImplementedError('TODO bước 3: cap_nhat')

def tinh_J(X, nhan, C):
    raise NotImplementedError('TODO bước 3: tinh_J')

def lloyd(X, C0, max_iter=300):
    # TODO: gán; lặp: cập nhật, gán lại; dừng khi nhãn không đổi; ghi J sau mỗi nửa bước
    raise NotImplementedError('TODO bước 3: lloyd')

In [ ]:
# TỰ KIỂM · bước 3
P = np.array([[1, 1], [2, 1], [3, 3], [5, 4], [6, 5], [7, 5]], float)
nhan6, C6, ls6 = lloyd(P, P[[0, 2]])
kiem('sáu điểm: J = 39 → 12 → 93/8 → 22/3', np.allclose(ls6[:4], [39, 12, 93 / 8, 22 / 3]), 'xem lại slide 20–24')
vt = np.random.RandomState(0).choice(len(X), 4, replace=False)
nhan, C, ls = lloyd(X, X[vt])
sk = KMeans(n_clusters=4, init=X[vt], n_init=1, tol=0, algorithm='lloyd').fit(X)
kiem('trùng KMeans (cùng khởi tạo, tol = 0) cho mọi khách', (sk.labels_ == nhan).all(), 'quy tắc hoà, điều kiện dừng')
kiem('J không tăng ở mọi nửa bước', all(b <= a + 1e-9 for a, b in zip(ls, ls[1:])))
print('số nửa bước:', len(ls), '· J cuối:', round(ls[-1], 3))

## Bước 4 (khá) · Chọn K bằng ba căn cứ

Với K = 2…6 (KMeans, `n_init=20, random_state=42`): inertia, silhouette trên **toàn bộ** khách, và ổn định bootstrap (10 lần: lấy mẫu có hoàn lại, fit, gán toàn bộ khách, so ARI với phân hoạch trên dữ liệu đủ). Lập bảng `bang_K` với các cột `K, inertia, silhouette, on_dinh`.

In [ ]:
# TODO: tạo bang_K (DataFrame) với các cột K, inertia, silhouette, on_dinh
raise NotImplementedError('TODO bước 4: bang_K')
print(bang_K.round(4).to_string(index=False))

In [ ]:
# TỰ KIỂM · bước 4
kiem('silhouette K = 2 = 0.4492', abs(float(bang_K.loc[bang_K.K == 2, 'silhouette'].iloc[0]) - 0.449231) < 1e-6, 'silhouette_score(X, nhãn) trên toàn bộ khách')
kiem('inertia không tăng theo K', (np.diff(bang_K.inertia) <= 1e-6).all())

## Bước 5 · Hồ sơ K = 4

Fit `KMeans(n_clusters=4, n_init=20, random_state=42, algorithm='lloyd')` trên X. Hồ sơ mỗi cụm: số khách, trung vị R, F, M trên **thang gốc**, phần của tổng M. Đặt tên mô tả cho từng cụm (không xếp hạng).

In [ ]:
# TODO: m4 = KMeans(...).fit(X); ho_so = bảng theo cụm: so_khach, R, F, M (trung vị), phan_M
raise NotImplementedError('TODO bước 5: m4 và ho_so')
print(ho_so.round(3))

In [ ]:
# TỰ KIỂM · bước 5
kiem('cỡ bốn cụm = [478, 480, 983, 1429]', sorted(ho_so.so_khach.tolist()) == [478, 480, 983, 1429], 'n_init = 20, random_state = 42, algorithm = "lloyd"')

## Báo cáo một trang (nộp cùng notebook)

1. **Dữ liệu và biểu diễn.** Mốc, số khách, định nghĩa R, F, M; vì sao log1p và chuẩn hoá (dẫn tỷ trọng phương sai).
2. **Thuật toán.** Hai nửa bước; bằng chứng mã tự viết trùng KMeans; J không tăng.
3. **Chọn K.** Bảng ba căn cứ (mức khá); lựa chọn có điều kiện, kèm đánh đổi.
4. **Hồ sơ và đề xuất.** Bốn hồ sơ có đơn vị; mỗi đề xuất là một giả thuyết kèm cách thử nghiệm (có nhóm đối chứng).
5. **Giới hạn.** Ít nhất ba: không có nhãn đúng; M chưa trừ hàng trả; một nhà bán lẻ, một năm; cụm đổi theo thời gian; kết quả phụ thuộc thang đo và khởi tạo.

## Nâng cao (chọn một)

- **A · k-means++ tự viết.** Viết `kmeanspp(X, K, rng)` (tâm đầu ngẫu nhiên, tâm sau rút với xác suất ∝ D²). Trên sáu điểm, K = 3: chạy 2.000 lần `kmeanspp` + `lloyd`, đếm tỷ lệ dừng ở J = 19/6; so với chọn đều 3 điểm (40%). Kết quả nên gần 54% (với 2.000 lần, sai số chuẩn khoảng 1,1 điểm phần trăm, nên khoảng 52–56% là hợp lý). Không dùng `KMeans(init='k-means++')` để thay: scikit-learn dùng biến thể tham lam, cho tỷ lệ cao hơn (khoảng 75% trên sáu điểm này).
- **B · Khách đổi cụm.** Tính RFM tại 10/12/2011 cho **cùng** các khách của tháng 9; biến đổi bằng **scaler của tháng 9**; gán bằng **tâm của `m4`**. Lập bảng chuyển cụm tháng 9 → tháng 12; tỷ lệ giữ cụm nên khoảng 61,8%. Giải thích vì sao nhiều khách “trôi” sang cụm lâu chưa mua.

In [ ]:
# Nâng cao B · khởi đầu (tuỳ chọn)
rfm12 = tinh_rfm(dong, pd.Timestamp('2011-12-10')).loc[rfm.index]
nhan12 = m4.predict(sc.transform(np.log1p(rfm12)))
print(pd.crosstab(pd.Series(m4.labels_, name='tháng 9'), pd.Series(nhan12, name='tháng 12')))
print('giữ cụm:', round(float(np.mean(nhan12 == m4.labels_)), 4))